# CrossVul-DG real-data execution (Colab/Kaggle)

Upload the local `crossvul-source.tar.gz` and authentic MegaVul release yourself. This notebook performs no account upload. Enable an available GPU, then inspect its actual resources. The bundle contains uncommitted local fixes that are absent from the remote repository. Do not clone the old remote checkout as a substitute.


In [ ]:
from pathlib import Path
import os, sys, subprocess, tarfile
# Colab: mount Drive yourself if needed. Kaggle: set these to attached input paths.
SOURCE_BUNDLE = Path("/content/drive/MyDrive/CrossVul-DG/crossvul-source.tar.gz")
RAW_DIR = Path("/content/drive/MyDrive/CrossVul-DG/raw/megavul")
PROCESSED_DIR = Path("/content/drive/MyDrive/CrossVul-DG/processed")
RUNS_DIR = Path("/content/drive/MyDrive/CrossVul-DG/runs")
WORK_DIR = Path("/content/crossvul-dg")
WORK_DIR.mkdir(exist_ok=True)
with tarfile.open(SOURCE_BUNDLE) as archive:
    archive.extractall(WORK_DIR, filter="data")
os.chdir(WORK_DIR)


In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before training"
print(torch.__version__, torch.cuda.get_device_name(), torch.cuda.mem_get_info())
# Preserve the platform's working CUDA torch. Install the remaining runtime dependencies.
requirements = [line for line in Path("requirements.txt").read_text().splitlines() if line and not line.startswith("torch")]
subprocess.run([sys.executable, "-m", "pip", "install", *requirements, "pytest"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_XET="1")
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


In [ ]:
subprocess.run([sys.executable,"scripts/prepare_data.py","--raw_dir",str(RAW_DIR),"--out_dir",str(PROCESSED_DIR),"--dedup"],check=True)
subprocess.run([sys.executable,"scripts/audit_data.py","--data_path",str(PROCESSED_DIR/"top10.parquet")],check=True)


## Full-population baseline

Both methods use exactly the same source split, held-out project and seed. No compute caps are used here. Start at length 128 with mixed precision and gradient checkpointing; resource preflight may reject unsupported configurations. Save runs to persistent storage. Thresholds and checkpoint selection use source validation only.


In [ ]:
HELD_OUT="torvalds/linux"  # preregistered largest project, selected by row count, not scores
for method in ("erm","cc-dro"):
    command=[sys.executable,"scripts/run_lopo.py","--config","configs/local_gpu.yaml","--method",method,
             "--held_out",HELD_OUT,"--seed","0","--data_path",str(PROCESSED_DIR/"top10.parquet"),
             "--output_dir",str(RUNS_DIR/"full_baselines"/method)]
    subprocess.run(command,check=True)


## Validation-only ablations

Print a one-variable-at-a-time plan for lambda `{0,.01,.1,1}`, LR `{1e-5,2e-5,3e-5}` and length `{128,256,512}`. Execute compatible jobs sequentially after resource checks, preserving all printed flags. `--selection_only` prevents test forward passes. Freeze the maximum-validation-AP configuration before evaluating its test checkpoint. Length 512 is conditional on GPU capacity.


In [ ]:
subprocess.run([sys.executable,"scripts/plan_real_ablations.py","--held_out",HELD_OUT,
                "--data_path",str(PROCESSED_DIR/"top10.parquet"),"--out_root",str(RUNS_DIR/"ablations")],check=True)
# After running selected printed jobs:
# subprocess.run([sys.executable,"scripts/select_validation_run.py","--runs_dir",str(RUNS_DIR/"ablations")],check=True)
# Re-run the chosen original command without --selection_only; its matching completed
# checkpoint is restored, and no extra training is performed before test evaluation.


In [ ]:
subprocess.run([sys.executable,"scripts/aggregate_results.py","--runs_dir",str(RUNS_DIR/"full_baselines"),
                "--out",str(RUNS_DIR/"tables"),"--expected_folds","1","--expected_seeds","1",
                "--expected_methods","erm","cc-dro"],check=True)
